In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import optuna
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer



from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

sys.path.append(str(Path().resolve().parent))
import src.data_processing.data_processing as dp
import src.data_processing.building_dataset as bd

/mnt/d/ml_project/ml_project_2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = dp.load_data('../data/raw/grades.csv')





/mnt/d/ml_project/ml_project_2026/src/data_processing/data_processing.py:23: DtypeWarning: Columns (0: course, 1: module, 2: academic_year) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep = ';')


In [4]:
df = dp.clean_data(df)
df = dp.process_data(df)

df_final = bd.make_features_for_status(df)
df_final.head()

,student_id_hash,avg_grade,count_grades,proportion_retake,proportion_retake_com,student_status,course_1,course_2,course_3,course_4,...,place_type_Внеконкурсное поступление,place_type_Коммерческие,place_type_Коммерческие за счет средств вуза,place_type_Коммерческие за счет средств вуза для иностранных граждан,place_type_Коммерческие места для иностранных граждан,place_type_Места с оплатой стоимости обучения на договорной основе,"place_type_Места, обеспеченные государственным финансированием",place_type_По межправительственным соглашениям,place_type_Сетевой студент,place_type_Целевые
0,00000d86dad0d3c197341f776bb61a1d,8.166667,24,0.000000,0.000000,graduated,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0001dd443aff5c0800694730323c3d7d,6.973913,115,0.034783,0.008696,study,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,00035b28bedc6d3a0336cb61a841d79b,7.214286,14,0.000000,0.000000,study,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,00044a704df898b9a87faf40795d62b6,8.454545,22,0.000000,0.000000,graduated,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0004d03d5c41e468a521ed8a964fe889,7.250000,16,0.062500,0.000000,study,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df_final = dp.str_to_int_student_status(df_final)

In [7]:
X = df_final.drop(columns=['student_id_hash', 'student_status'])
y = df_final['student_status']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train_mini = np.concat((X_train[y_train==0][:20*y_train.sum()], X_train[y_train==1]))
y_train_mini = np.concat((y_train[y_train==0][:20*y_train.sum()], y_train[y_train==1]))

In [12]:

def objective(trial, X, y):
    params = {

        'max_depth': trial.suggest_int('max_depth', 2, 30),

        'min_samples_split': trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),

        'min_samples_leaf': trial.suggest_int(
            'min_samples_leaf',
            1,
            20
        ),

        'criterion': trial.suggest_categorical(
            'criterion',
            ['gini', 'entropy', 'log_loss']
        ),

        'max_features': trial.suggest_categorical(
            'max_features',
            [None, 'sqrt', 'log2']
        )
    }

    model = DecisionTreeClassifier(
        **params,
        random_state=42
    )

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1'
    ).mean()

    return score


def tune_decision_tree(
    X,
    y,
    n_trials: int = 50
):


    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective(trial, X, y),
        n_trials=n_trials
    )

    best_params = study.best_params

    print("Лучшие параметры")
    print(best_params)

    print()
    print(f"Лучший score: {study.best_value:.4f}")

    best_model = DecisionTreeClassifier(
        **best_params,
        random_state=42
    )

    best_model.fit(X, y)

    return best_model, study
best_model_DecisionTree, model_DecisionTree = tune_decision_tree(
    X_train,
    y_train,
    n_trials=100
)

[I 2026-05-18 14:23:01,126] A new study created in memory with name: no-name-b1ef27c9-7253-43f1-aed4-909dc5e26d15
[I 2026-05-18 14:23:03,419] Trial 0 finished with value: 0.6150869730860652 and parameters: {'max_depth': 23, 'min_samples_split': 7, 'min_samples_leaf': 13, 'criterion': 'log_loss', 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6150869730860652.
[I 2026-05-18 14:23:04,975] Trial 1 finished with value: 0.5901855623927886 and parameters: {'max_depth': 26, 'min_samples_split': 2, 'min_samples_leaf': 7, 'criterion': 'gini', 'max_features': 'log2'}. Best is trial 0 with value: 0.6150869730860652.
[I 2026-05-18 14:23:06,672] Trial 2 finished with value: 0.5849741124338033 and parameters: {'max_depth': 26, 'min_samples_split': 15, 'min_samples_leaf': 5, 'criterion': 'entropy', 'max_features': 'log2'}. Best is trial 0 with value: 0.6150869730860652.
[I 2026-05-18 14:23:07,771] Trial 3 finished with value: 0.0 and parameters: {'max_depth': 3, 'min_samples_split': 10, 'min_

Лучшие параметры
{'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 2, 'criterion': 'entropy', 'max_features': None}

Лучший score: 0.8215


In [14]:



def objective_rf(trial, X, y):


    params = {
        'n_estimators': trial.suggest_int(
            'n_estimators',
            50,
            200
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            2,
            20
        ),

        'min_samples_split': trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),

        'min_samples_leaf': trial.suggest_int(
            'min_samples_leaf',
            1,
            20
        ),

        'max_features': trial.suggest_categorical(
            'max_features',
            ['sqrt', 'log2', None]
        ),

        'bootstrap': trial.suggest_categorical(
            'bootstrap',
            [True, False]
        )
    }

    model = RandomForestClassifier(
        **params,
        class_weight='balanced',
        random_state=42,
        n_jobs=1
    )

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    return score


def tune_random_forest(
    X,
    y,
    n_trials: int = 50
):

    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective_rf(trial, X, y),
        n_trials=n_trials
    )

    best_params = study.best_params


    best_model = RandomForestClassifier(
        **best_params,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    best_model.fit(X, y)

    return best_model, study

best_rf, study_rf = tune_random_forest(
    X_train,
    y_train,
    n_trials=50
)

[I 2026-05-18 14:48:17,950] A new study created in memory with name: no-name-0b76c3e4-6c2d-432a-b08d-b8e6e8c855ec
[I 2026-05-18 14:48:32,303] Trial 0 finished with value: 0.6210537372628276 and parameters: {'n_estimators': 137, 'max_depth': 4, 'min_samples_split': 15, 'min_samples_leaf': 20, 'max_features': 'log2', 'bootstrap': False}. Best is trial 0 with value: 0.6210537372628276.
[I 2026-05-18 14:49:05,659] Trial 1 finished with value: 0.6299478438842552 and parameters: {'n_estimators': 165, 'max_depth': 19, 'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False}. Best is trial 1 with value: 0.6299478438842552.
[I 2026-05-18 14:49:26,098] Trial 2 finished with value: 0.6224597239149399 and parameters: {'n_estimators': 156, 'max_depth': 16, 'min_samples_split': 18, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}. Best is trial 1 with value: 0.6299478438842552.
[I 2026-05-18 14:50:17,944] Trial 3 finished with value: 0.601364656713

In [15]:

def objective_gb(trial, X, y):


    params = {

        'n_estimators': trial.suggest_int(
            'n_estimators',
            50,
            200
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.001,
            0.3,
            log=True
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            2,
            10
        ),

        'min_samples_split': trial.suggest_int(
            'min_samples_split',
            2,
            20
        ),

        'min_samples_leaf': trial.suggest_int(
            'min_samples_leaf',
            1,
            20
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.5,
            1.0
        ),

        'max_features': trial.suggest_categorical(
            'max_features',
            ['sqrt', 'log2', None]
        )
    }

    model = GradientBoostingClassifier(
        **params,
        random_state=42
    )

    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1',
        n_jobs=-1
    ).mean()

    return score


def tune_gradient_boosting(
    X,
    y,
    n_trials: int = 50
):


    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective_gb(trial, X, y),
        n_trials=n_trials
    )

    best_params = study.best_params


    best_model = GradientBoostingClassifier(
        **best_params,
        random_state=42
    )

    best_model.fit(X, y)

    return best_model, study
best_gb, study_gb = tune_gradient_boosting(
    X_train,
    y_train,
    n_trials=100
)

[I 2026-05-18 15:22:36,322] A new study created in memory with name: no-name-a912ab08-9a92-4479-aafe-0631529687c4
[I 2026-05-18 15:24:52,019] Trial 0 finished with value: 0.8256227461805643 and parameters: {'n_estimators': 105, 'learning_rate': 0.05114255856418449, 'max_depth': 7, 'min_samples_split': 13, 'min_samples_leaf': 4, 'subsample': 0.5278507513112994, 'max_features': None}. Best is trial 0 with value: 0.8256227461805643.
[I 2026-05-18 15:25:03,122] Trial 1 finished with value: 0.0 and parameters: {'n_estimators': 163, 'learning_rate': 0.00137256518014375, 'max_depth': 5, 'min_samples_split': 14, 'min_samples_leaf': 20, 'subsample': 0.6089946709540119, 'max_features': 'log2'}. Best is trial 0 with value: 0.8256227461805643.
[I 2026-05-18 15:25:26,245] Trial 2 finished with value: 0.8236785849941242 and parameters: {'n_estimators': 186, 'learning_rate': 0.2362704989023102, 'max_depth': 5, 'min_samples_split': 18, 'min_samples_leaf': 8, 'subsample': 0.8496202389693928, 'max_featu

In [ ]:

models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth = 6, min_samples_split =  4, min_samples_leaf = 2, criterion = 'entropy', max_features = None),
    'Random Forest': RandomForestClassifier(n_estimators = 109, max_depth =  16, min_samples_split =  5, min_samples_leaf = 1, max_features = 'log2', bootstrap = True),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators = 108, learning_rate = 0.03736563332951993,  max_depth = 8, min_samples_split = 13, min_samples_leaf= 6, subsample= 0.6816427025399264, max_features = None)
}

results = []

for name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)



    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Precision_w': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred),
        'Recall_w': recall_score(y_test, y_pred, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred),
        'F1-Score_w': f1_score(y_test, y_pred, average='weighted')
    })

results_df = pd.DataFrame(results)
print(results_df.round(4))

best_model = results_df.loc[results_df['F1-Score'].idxmax(), 'Model']
print(f"\nЛучшая модель по F1-Score: {best_model}")

                 Model  Accuracy  Precision  Precision_w  Recall  Recall_w  \
0  Logistic Regression    0.9893     0.9072       0.9888  0.7370    0.9893   
1        Decision Tree    0.9900     0.9082       0.9896  0.7621    0.9900   
2        Random Forest    0.9738     1.0000       0.9745  0.1725    0.9738   
3    Gradient Boosting    0.9908     0.9029       0.9904  0.7940    0.9908   

   F1-Score  F1-Score_w  
0    0.8133      0.9888  
1    0.8288      0.9896  
2    0.2943      0.9648  
3    0.8449      0.9905  

Лучшая модель по F1-Score: Gradient Boosting


In [16]:
results_df

,Model,Accuracy,Precision,Precision_w,Recall,Recall_w,F1-Score,F1-Score_w
0,Logistic Regression,0.989300,0.907216,0.988800,0.737018,0.989300,0.813309,0.988763
1,Decision Tree,0.990042,0.908184,0.989614,0.762144,0.990042,0.828780,0.989620
2,Random Forest,0.973833,1.000000,0.974522,0.172529,0.973833,0.294286,0.964775
3,Gradient Boosting,0.990783,0.902857,0.990438,0.793970,0.990783,0.844920,0.990497
